In [4]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
===============================================================================
JCP FINAL v6.0.3 (PATCHED)

===============================================================================
"""

import os
import json
import time
import math
import logging
import dataclasses
from typing import List, Dict, Tuple, Optional
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.autograd as autograd
from tqdm import tqdm

# ==============================================================================
# CONFIGURATION MANAGEMENT
# ==============================================================================
@dataclasses.dataclass
class ExperimentConfig:
    """实验配置 - 支持多时间尺度和多参数设置"""
    T_FINAL: float = 1.0
    X_MIN: float = -10.0
    X_MAX: float = 10.0
    NUM_SEEDS: int = 5
    SEEDS: List[int] = None
    N_TRAIN_COLLOC: int = 4000
    N_IC_TRAIN: int = 256
    N_INV_QUAD: int = 256
    PROJ_NX: int = 512
    ENABLE_SPECTRAL: bool = True
    N_SPECTRAL_PTS: int = 10
    N_TRAJECTORY_PTS: int = 20
    RESAMPLE_INTERVAL: int = 300
    RESAMPLE_RATIO: float = 0.4
    N_TS_INVARIANT: int = 6
    PROJ_CACHE_DECIMALS: int = 6
    PROJ_CACHE_MAXSIZE: int = 20000

    def __post_init__(self):
        if self.SEEDS is None:
            self.SEEDS = list(range(42, 42 + self.NUM_SEEDS))


# 快速测试配置（500次迭代）
CONFIG = ExperimentConfig(
    T_FINAL=1.0,
    NUM_SEEDS=5,
    ENABLE_SPECTRAL=True,
)

# 完整发表配置（取消注释使用）
# CONFIG = ExperimentConfig(
#     T_FINAL=1.0,
#     NUM_SEEDS=5,
#     ENABLE_SPECTRAL=True,
# )
# ITERATIONS = 5000
# LOG_INTERVAL = 200
# SAVE_CHECKPOINT_INTERVAL = 500

RESULT_DIR = "jcp_results_v6_0_3"
CHECKPOINT_DIR = os.path.join(RESULT_DIR, "checkpoints")
FIELDS_DIR = os.path.join(RESULT_DIR, "fields")
CURVES_DIR = os.path.join(RESULT_DIR, "curves")

# 保持500次快速测试设置
LOG_INTERVAL = 200
SAVE_CHECKPOINT_INTERVAL = 400
ITERATIONS = 2000
LR_MILESTONES = [800, 1400, 1800]
LR_GAMMA = 0.3
WARMUP_ITERS = 60

MLP_IN_DIM = 2
MLP_HIDDEN = 128
MLP_DEPTH = 4
ACTIVATION = "tanh"

LEARNING_RATE = 1e-3

N_CAUSAL_WINDOWS = 10
PDE_CAUSAL_TAU = {
    "KdV": 1.2, "Burgers": 0.7, "NLS": 0.6, "KS": 2.5,
}

N_CONSERVATION_PTS = 40

MU_PENALTY = 3.0
IC_WEIGHT_VANILLA = 100.0
LAMBDA_FIXED_IC = 8.0
FIXED_LAMBDA_INV = {"KdV": 0.08, "Burgers": 0.008, "NLS": 0.008, "KS": 0.08}
SA_INIT = 0.8
SA_LR = 5e-4

KDV_ALPHA_CLAMP = 5.0
KS_ALPHA_CLAMP = 5.0
NLS_SCALE_MIN = 0.1
NLS_SCALE_MAX = 10.0
NEWTON_MAX_ITER = 20
NEWTON_TOL = 1e-10

NX_REF = 1024
DT_REF_DEFAULT = 1e-4
DT_REF_KS = 5e-5
SAVE_EVERY = 20
N_RESID_TEST = 8000

PDE_LIST = [
    ("KdV", 1, "conservative"),
    ("Burgers", 1, "dissipative"),
    ("NLS", 2, "conservative"),
    ("KS", 1, "mean_preserving"),
]

# ==============================================================================
# REPRODUCIBILITY & SETUP
# ==============================================================================
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs(RESULT_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(FIELDS_DIR, exist_ok=True)
os.makedirs(CURVES_DIR, exist_ok=True)

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)-8s %(message)s")
log = logging.getLogger("JCP-FINAL-v6.0.2-PATCHED")


def set_seed(seed: int):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


# ==============================================================================
# MLP & AUTOGRAD HELPER
# ==============================================================================
class MLP(nn.Module):
    def __init__(self, out_dim: int = 1):
        super().__init__()
        gain = 5.0 / 3.0 if ACTIVATION == "tanh" else 1.0
        act_cls = {"tanh": nn.Tanh, "swish": nn.SiLU}[ACTIVATION]
        layers = [nn.Linear(MLP_IN_DIM, MLP_HIDDEN), act_cls()]
        for _ in range(MLP_DEPTH - 1):
            layers += [nn.Linear(MLP_HIDDEN, MLP_HIDDEN), act_cls()]
        layers.append(nn.Linear(MLP_HIDDEN, out_dim))
        self.net = nn.Sequential(*layers)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight, gain=gain)
                nn.init.zeros_(m.bias)

    def forward(self, xt: torch.Tensor) -> torch.Tensor:
        return self.net(xt)


def grad(u: torch.Tensor, x: torch.Tensor, retain_graph=True) -> torch.Tensor:
    return autograd.grad(
        u, x,
        grad_outputs=torch.ones_like(u),
        create_graph=True,
        retain_graph=retain_graph,
    )[0]


# ==============================================================================
# PDE LIBRARY (ALL BUGS FIXED)
# ==============================================================================
class PDE:
    NU = 0.01

    @staticmethod
    def kdv_res(u, xt):
        g = grad(u, xt, retain_graph=True)
        ux, ut = g[:, [0]], g[:, [1]]
        uxx = grad(ux, xt, retain_graph=True)[:, [0]]
        uxxx = grad(uxx, xt, retain_graph=True)[:, [0]]
        return ut + 6.0 * u * ux + uxxx

    @staticmethod
    def kdv_ic(x):
        return 0.5 / torch.cosh(x / 2.0) ** 2

    @staticmethod
    def kdv_I(u, xt):
        ux = grad(u, xt, retain_graph=True)[:, [0]]
        integrand = 0.5 * ux**2 - u**3
        x = xt[:, [0]]
        return torch.trapezoid(integrand.squeeze(), x.squeeze())

    @staticmethod
    def kdv_I_ic(x: torch.Tensor) -> float:
        u = PDE.kdv_ic(x).squeeze()
        xx = x.squeeze()
        dx = (xx[1] - xx[0]).item()
        ux = torch.gradient(u, spacing=(dx,))[0]
        integrand = 0.5 * ux**2 - u**3
        I0 = torch.trapezoid(integrand, xx)
        return float(I0.item())

    @staticmethod
    def burgers_res(u, xt):
        g = grad(u, xt, retain_graph=True)
        ux, ut = g[:, [0]], g[:, [1]]
        uxx = grad(ux, xt, retain_graph=True)[:, [0]]
        return ut + u * ux - PDE.NU * uxx

    @staticmethod
    def burgers_ic(x):
        return -torch.sin(np.pi * x / 10.0)

    @staticmethod
    def burgers_E(u, xt):
        x = xt[:, [0]]
        return 0.5 * torch.trapezoid(u.squeeze() ** 2, x.squeeze())

    @staticmethod
    def burgers_D(u, xt):
        ux = grad(u, xt, retain_graph=True)[:, [0]]
        x = xt[:, [0]]
        return PDE.NU * torch.trapezoid(ux.squeeze() ** 2, x.squeeze())

    @staticmethod
    def nls_res(u, xt):
        ur, ui = u[:, [0]], u[:, [1]]
        gr = grad(ur, xt, retain_graph=True)
        gi = grad(ui, xt, retain_graph=True)
        urx, urt = gr[:, [0]], gr[:, [1]]
        uix, uit = gi[:, [0]], gi[:, [1]]
        uxxr = grad(urx, xt, retain_graph=True)[:, [0]]
        uxxi = grad(uix, xt, retain_graph=True)[:, [0]]
        a2 = ur**2 + ui**2
        # ✅ FIXED: NLS残差符号（聚焦NLS）- 审稿人确认
        return torch.cat([
            uit - 0.5 * uxxr - a2 * ur,
            -urt - 0.5 * uxxi - a2 * ui,
        ], dim=1)

    @staticmethod
    def nls_ic(x):
        return torch.exp(-x**2 / 2.0)

    @staticmethod
    def nls_I(u, xt):
        integrand = u[:, 0]**2 + u[:, 1]**2
        x = xt[:, [0]]
        return torch.trapezoid(integrand.squeeze(), x.squeeze())

    @staticmethod
    def nls_I_ic(x: torch.Tensor) -> float:
        u = PDE.nls_ic(x).squeeze()
        xx = x.squeeze()
        integrand = u**2
        I0 = torch.trapezoid(integrand, xx)
        return float(I0.item())

    @staticmethod
    def ks_res(u, xt):
        g = grad(u, xt, retain_graph=True)
        ux, ut = g[:, [0]], g[:, [1]]
        uxx = grad(ux, xt, retain_graph=True)[:, [0]]
        uxxx = grad(uxx, xt, retain_graph=True)[:, [0]]
        uxxxx = grad(uxxx, xt, retain_graph=True)[:, [0]]
        return ut + u * ux + uxx + uxxxx

    @staticmethod
    def ks_ic(x):
        return torch.cos(np.pi * x / 16.0)

    @staticmethod
    def ks_I(u, xt):
        x = xt[:, [0]]
        return torch.trapezoid(u.squeeze(), x.squeeze())

    @staticmethod
    def ks_I_ic(x: torch.Tensor) -> float:
        u = PDE.ks_ic(x).squeeze()
        xx = x.squeeze()
        I0 = torch.trapezoid(u, xx)
        return float(I0.item())


PDE_REGISTRY = {
    "KdV": (PDE.kdv_res, PDE.kdv_ic, PDE.kdv_I),
    "Burgers": (PDE.burgers_res, PDE.burgers_ic, None),
    "NLS": (PDE.nls_res, PDE.nls_ic, PDE.nls_I),
    "KS": (PDE.ks_res, PDE.ks_ic, PDE.ks_I),
}


# ==============================================================================
# SAMPLING & LOSS FUNCTIONS
# ==============================================================================
def sample_colloc(n, t_final=None):
    if t_final is None:
        t_final = CONFIG.T_FINAL
    x = torch.rand(n, 1, device=DEVICE) * (CONFIG.X_MAX - CONFIG.X_MIN) + CONFIG.X_MIN
    t = torch.rand(n, 1, device=DEVICE) * t_final
    return x, t


def sample_ic(n):
    return torch.linspace(CONFIG.X_MIN, CONFIG.X_MAX, n, device=DEVICE).unsqueeze(1)


def quad_grid(nx, requires_grad=False):
    g = torch.linspace(CONFIG.X_MIN, CONFIG.X_MAX, nx, device=DEVICE).unsqueeze(1)
    if requires_grad:
        return g.requires_grad_(True)
    return g


def time_slices(n, t_final=None):
    if t_final is None:
        t_final = CONFIG.T_FINAL
    return torch.linspace(0.0, t_final, n, device=DEVICE)


def causal_pde_loss(model, x_f, t_f, residual_fn, tau):
    tau_eff = tau / (CONFIG.T_FINAL + 1e-12)
    
    edges = torch.linspace(0.0, CONFIG.T_FINAL, N_CAUSAL_WINDOWS + 1, device=DEVICE)
    t_sq = t_f.squeeze()
    win_losses = []
    for k in range(N_CAUSAL_WINDOWS):
        mask = (t_sq >= edges[k]) & (t_sq < edges[k + 1])
        if mask.sum() == 0:
            win_losses.append(torch.tensor(0.0, device=DEVICE))
            continue
        xt = torch.cat([x_f[mask], t_f[mask]], dim=1).requires_grad_(True)
        r = residual_fn(model(xt), xt)
        win_losses.append(torch.mean(r**2))
    Lw = torch.stack(win_losses)
    cumsum = torch.zeros_like(Lw)
    for k in range(1, N_CAUSAL_WINDOWS):
        cumsum[k] = cumsum[k - 1] + Lw[k - 1].detach()
    weights = torch.exp(-tau_eff * cumsum)
    return (weights * Lw).sum(), Lw.detach()


def resample_collocation(model, x_f, t_f, residual_fn):
    model.eval()
    n = x_f.shape[0]
    n_rep = int(n * CONFIG.RESAMPLE_RATIO)
    with torch.enable_grad():
        xt = torch.cat([x_f, t_f], dim=1).requires_grad_(True)
        r = residual_fn(model(xt), xt)
        mag = (r**2).mean(dim=1).detach()
        _, keep = torch.topk(mag, n - n_rep, largest=True)
        x_new = torch.rand(n_rep, 1, device=DEVICE) * (CONFIG.X_MAX - CONFIG.X_MIN) + CONFIG.X_MIN
        t_new = torch.rand(n_rep, 1, device=DEVICE) * CONFIG.T_FINAL
    model.train()
    return (
        torch.cat([x_f[keep].detach(), x_new], dim=0),
        torch.cat([t_f[keep].detach(), t_new], dim=0),
    )


# ==============================================================================
# CONSTRAINT EVALUATION
# ==============================================================================
def constraint_conservative(model, ts, inv_fn, inv0, nx=None):
    if nx is None:
        nx = CONFIG.N_INV_QUAD
    norm = abs(inv0) + 1e-12
    cs = []
    for t in ts:
        x = quad_grid(nx, requires_grad=False)
        xt = torch.cat([x, t.expand_as(x)], dim=1).requires_grad_(True)
        u = model(xt)
        I = inv_fn(u, xt)
        cs.append((I - inv0) / norm)
    return torch.stack(cs)


def constraint_burgers(model, ts, inv0, nx=None):
    if nx is None:
        nx = CONFIG.N_INV_QUAD
    norm = abs(inv0) + 1e-12
    cs = []
    for t in ts:
        t_leaf = t.clone().detach().requires_grad_(True)
        x = quad_grid(nx, requires_grad=False)
        xt = torch.cat([x, t_leaf.expand_as(x)], dim=1).requires_grad_(True)
        u = model(xt)
        E = PDE.burgers_E(u, xt)
        D = PDE.burgers_D(u, xt)
        # CRITICAL: 训练时必须 create_graph=True，否则约束项无法反向传播到网络参数
        dEdt = autograd.grad(E, t_leaf, create_graph=True)[0]
        cs.append((dEdt + D) / norm)
    return torch.stack(cs)


def ic_loss(model, x_ic, ic_fn, out_dim):
    xt = torch.cat([x_ic, torch.zeros_like(x_ic)], dim=1)
    u0 = model(xt)
    gt = ic_fn(x_ic)
    if out_dim == 2:
        gt = torch.cat([gt, torch.zeros_like(gt)], dim=1)
    return torch.mean((u0 - gt) ** 2)


# ==============================================================================
# LOSS RECORD
# ==============================================================================
@dataclasses.dataclass
class LossRecord:
    Lp: List[float] = dataclasses.field(default_factory=list)
    Lic: List[float] = dataclasses.field(default_factory=list)
    Cinv: List[float] = dataclasses.field(default_factory=list)
    Linv: List[float] = dataclasses.field(default_factory=list)
    Ltot: List[float] = dataclasses.field(default_factory=list)
    lam_ic: List[float] = dataclasses.field(default_factory=list)
    lam_inv: List[float] = dataclasses.field(default_factory=list)


# ==============================================================================
# BASE PINN & METHODS
# ==============================================================================
class BasePINN(nn.Module):
    label = "base"
    def __init__(self, out_dim, pde_type):
        super().__init__()
        self.out_dim = out_dim
        self.pde_type = pde_type
        self.net = MLP(out_dim)

    def forward(self, xt):
        return self.net(xt)

    def compute_loss(self, *args, **kwargs):
        raise NotImplementedError


class VanillaPINN(BasePINN):
    label = "Vanilla"
    def compute_loss(self, x_f, t_f, x_ic, ts, residual_fn, ic_fn, inv_fn, inv0, warmup, tau):
        Lp, _ = causal_pde_loss(self, x_f, t_f, residual_fn, tau)
        Lic = ic_loss(self, x_ic, ic_fn, self.out_dim)
        Ltot = Lp + IC_WEIGHT_VANILLA * Lic
        zero = torch.tensor(0.0, device=DEVICE)
        return Ltot, dict(Lp=Lp, Lic=Lic, Cinv=zero, Linv=zero, Ltot=Ltot)


class PenaltyPINN(BasePINN):
    label = "Penalty"
    def compute_loss(self, x_f, t_f, x_ic, ts, residual_fn, ic_fn, inv_fn, inv0, warmup, tau):
        Lp, _ = causal_pde_loss(self, x_f, t_f, residual_fn, tau)
        Lic = ic_loss(self, x_ic, ic_fn, self.out_dim)
        zero = torch.tensor(0.0, device=DEVICE)
        if warmup:
            Ltot = Lp + IC_WEIGHT_VANILLA * Lic
            return Ltot, dict(Lp=Lp, Lic=Lic, Cinv=zero, Linv=zero, Ltot=Ltot)
        if self.pde_type in ("conservative", "mean_preserving"):
            c_stack = constraint_conservative(self, ts, inv_fn, inv0)
        else:
            c_stack = constraint_burgers(self, ts, inv0)
        linv = 0.5 * MU_PENALTY * torch.mean(c_stack**2)
        Ltot = Lp + IC_WEIGHT_VANILLA * Lic + linv
        return Ltot, dict(
            Lp=Lp, Lic=Lic,
            Cinv=torch.mean(torch.abs(c_stack)).detach(),
            Linv=linv.detach(),
            Ltot=Ltot
        )


class FixedLambdaPINN(BasePINN):
    label = "Fixed-λ"
    def __init__(self, out_dim, pde_type, pde_name):
        super().__init__(out_dim, pde_type)
        self.lam_ic = float(LAMBDA_FIXED_IC)
        self.lam_inv = float(FIXED_LAMBDA_INV.get(pde_name, 0.1))

    def compute_loss(self, x_f, t_f, x_ic, ts, residual_fn, ic_fn, inv_fn, inv0, warmup, tau):
        Lp, _ = causal_pde_loss(self, x_f, t_f, residual_fn, tau)
        Lic = ic_loss(self, x_ic, ic_fn, self.out_dim)
        zero = torch.tensor(0.0, device=DEVICE)
        if warmup:
            Ltot = Lp + self.lam_ic * Lic + 0.5 * MU_PENALTY * Lic**2
            return Ltot, dict(Lp=Lp, Lic=Lic, Cinv=zero, Linv=zero, Ltot=Ltot)

        if self.pde_type in ("conservative", "mean_preserving"):
            c_stack = constraint_conservative(self, ts, inv_fn, inv0)
        else:
            c_stack = constraint_burgers(self, ts, inv0)

        linv = torch.mean(self.lam_inv * c_stack + 0.5 * MU_PENALTY * c_stack**2)
        Ltot = Lp + self.lam_ic * Lic + 0.5 * MU_PENALTY * Lic**2 + linv
        return Ltot, dict(
            Lp=Lp, Lic=Lic,
            Cinv=torch.mean(torch.abs(c_stack)).detach(),
            Linv=linv.detach(),
            Ltot=Ltot
        )


class SAPINN_Pos(BasePINN):
    label = "SA-PINN-pos"
    def __init__(self, out_dim, pde_type):
        super().__init__(out_dim, pde_type)
        self.log_lam_ic = nn.Parameter(torch.tensor(np.log(SA_INIT), dtype=torch.float32))
        self.log_lam_inv = nn.Parameter(torch.tensor(np.log(SA_INIT), dtype=torch.float32))

    @property
    def lam_ic(self): return self.log_lam_ic.exp().item()

    @property
    def lam_inv(self): return self.log_lam_inv.exp().item()

    def compute_loss(self, x_f, t_f, x_ic, ts, residual_fn, ic_fn, inv_fn, inv0, warmup, tau):
        lam_ic = self.log_lam_ic.exp()
        lam_inv = self.log_lam_inv.exp()
        Lp, _ = causal_pde_loss(self, x_f, t_f, residual_fn, tau)
        Lic = ic_loss(self, x_ic, ic_fn, self.out_dim)
        zero = torch.tensor(0.0, device=DEVICE)
        if warmup:
            Ltot = Lp + IC_WEIGHT_VANILLA * Lic
            return Ltot, dict(Lp=Lp, Lic=Lic, Cinv=zero, Linv=zero, Ltot=Ltot)

        if self.pde_type in ("conservative", "mean_preserving"):
            c_stack = constraint_conservative(self, ts, inv_fn, inv0)
        else:
            c_stack = constraint_burgers(self, ts, inv0)

        linv = torch.mean(lam_inv * torch.abs(c_stack) + 0.5 * MU_PENALTY * c_stack**2)
        Ltot = Lp + lam_ic * Lic + 0.5 * MU_PENALTY * Lic**2 + linv
        return Ltot, dict(
            Lp=Lp, Lic=Lic,
            Cinv=torch.mean(torch.abs(c_stack)).detach(),
            Linv=linv.detach(),
            Ltot=Ltot
        )


# ==============================================================================
# TWO-STAGE PROJECTION PINN (PATCHED)
# ==============================================================================
class TwoStageProjPINN(BasePINN):
    label = "TwoStage-Proj"

    def __init__(self, out_dim, pde_type, pde_name: str = "unknown"):
        super().__init__(out_dim, pde_type)
        self._pde_name = pde_name
        self._proj_enabled = False
        self._inv_fn = None
        self._inv0 = None
        self._L = float(CONFIG.X_MAX - CONFIG.X_MIN)
        self._cache: Dict[float, float] = {}

    def enable_projection(self, inv_fn, inv0):
        self._proj_enabled = True
        self._inv_fn = inv_fn
        self._inv0 = float(inv0)
        self._cache.clear()

    def disable_projection(self):
        self._proj_enabled = False
        self._inv_fn = None
        self._inv0 = None
        self._cache.clear()

    def _compute_kdv_alpha(self, t_val: float) -> float:
        t_key = round(t_val, CONFIG.PROJ_CACHE_DECIMALS)
        if t_key in self._cache:
            return self._cache[t_key]

        if len(self._cache) > CONFIG.PROJ_CACHE_MAXSIZE:
            self._cache.clear()

        x = quad_grid(CONFIG.PROJ_NX, requires_grad=False)
        t = torch.full_like(x, t_val)
        xt = torch.cat([x, t], dim=1).requires_grad_(True)

        with torch.enable_grad():
            u_raw = self.net(xt)
            ux = grad(u_raw, xt, retain_graph=True)[:, [0]]

            I_kinetic = torch.trapezoid(0.5 * ux.squeeze()**2, x.squeeze()).detach().item()
            M3 = torch.trapezoid(u_raw.squeeze()**3, x.squeeze()).detach().item()
            M2 = torch.trapezoid(u_raw.squeeze()**2, x.squeeze()).detach().item()
            M1 = torch.trapezoid(u_raw.squeeze(), x.squeeze()).detach().item()

        I0 = self._inv0
        L = self._L

        def F(a):
            return I_kinetic - M3 - 3*a*M2 - 3*a**2*M1 - a**3*L - I0

        def dF_da(a):
            return -3*M2 - 6*a*M1 - 3*a**2*L

        alpha = (I0 - (I_kinetic - M3)) / (-3*M2 + 1e-12)
        for _ in range(NEWTON_MAX_ITER):
            f = F(alpha)
            if abs(f) < NEWTON_TOL:
                alpha = float(np.clip(alpha, -KDV_ALPHA_CLAMP, KDV_ALPHA_CLAMP))
                self._cache[t_key] = alpha
                return alpha
            df = dF_da(alpha)
            if abs(df) < 1e-14:
                break
            alpha -= f / df
            # ✅ PATCH 3: Newton 迭代中间数值保护，防止越界导致 F(alpha) 爆炸
            alpha = float(np.clip(alpha, -KDV_ALPHA_CLAMP, KDV_ALPHA_CLAMP))

        if abs(F(alpha)) < NEWTON_TOL:
            alpha = float(np.clip(alpha, -KDV_ALPHA_CLAMP, KDV_ALPHA_CLAMP))
            self._cache[t_key] = alpha
            return alpha

        a_grid = torch.linspace(-KDV_ALPHA_CLAMP, KDV_ALPHA_CLAMP, 401, device=DEVICE)
        f_grid = (I_kinetic - M3
                  - 3*a_grid*M2
                  - 3*a_grid**2 * M1
                  - a_grid**3 * L
                  - I0)

        mask = f_grid[:-1] * f_grid[1:] <= 0
        idxs = torch.where(mask)[0]

        if len(idxs) > 0:
            idx = idxs[0].item()
            lo = a_grid[idx].item()
            hi = a_grid[idx + 1].item()

            lo_val = F(lo)
            hi_val = F(hi)

            if abs(lo_val) < NEWTON_TOL:
                alpha = lo
            elif abs(hi_val) < NEWTON_TOL:
                alpha = hi
            else:
                for _ in range(60):
                    mid = (lo + hi) / 2.0
                    if F(mid) * F(lo) > 0:
                        lo = mid
                    else:
                        hi = mid
                alpha = (lo + hi) / 2.0
        else:
            best_idx = torch.argmin(torch.abs(f_grid)).item()
            alpha = a_grid[best_idx].item()

        alpha = float(np.clip(alpha, -KDV_ALPHA_CLAMP, KDV_ALPHA_CLAMP))
        self._cache[t_key] = alpha
        return alpha

    def _compute_ks_alpha(self, t_val: float) -> float:
        t_key = round(t_val, CONFIG.PROJ_CACHE_DECIMALS)
        if t_key in self._cache:
            return self._cache[t_key]

        if len(self._cache) > CONFIG.PROJ_CACHE_MAXSIZE:
            self._cache.clear()

        x = quad_grid(CONFIG.PROJ_NX, requires_grad=False)
        t = torch.full_like(x, t_val)
        xt = torch.cat([x, t], dim=1)

        u_raw = self.net(xt)
        I_raw = torch.trapezoid(u_raw.squeeze(), x.squeeze()).item()

        alpha = (self._inv0 - I_raw) / (self._L + 1e-12)
        alpha = float(np.clip(alpha, -KS_ALPHA_CLAMP, KS_ALPHA_CLAMP))

        self._cache[t_key] = alpha
        return alpha

    def _compute_nls_scale(self, t_val: float) -> float:
        t_key = round(t_val, CONFIG.PROJ_CACHE_DECIMALS)
        if t_key in self._cache:
            return self._cache[t_key]

        if len(self._cache) > CONFIG.PROJ_CACHE_MAXSIZE:
            self._cache.clear()

        x = quad_grid(CONFIG.PROJ_NX, requires_grad=False)
        t = torch.full_like(x, t_val)
        xt = torch.cat([x, t], dim=1)

        u_raw = self.net(xt)
        I_raw = torch.trapezoid(u_raw[:, 0]**2 + u_raw[:, 1]**2, x.squeeze()).item()

        ratio = self._inv0 / (I_raw + 1e-12)
        ratio = max(ratio, 0.0)
        s = math.sqrt(ratio + 1e-12)
        s = float(np.clip(s, NLS_SCALE_MIN, NLS_SCALE_MAX))

        self._cache[t_key] = s
        return s

    def forward_raw(self, xt: torch.Tensor) -> torch.Tensor:
        return self.net(xt)

    def forward_proj(self, xt: torch.Tensor) -> torch.Tensor:
        u_raw = self.net(xt)

        if not self._proj_enabled or self.pde_type not in ("conservative", "mean_preserving"):
            return u_raw

        scale = float(10**CONFIG.PROJ_CACHE_DECIMALS)
        t_keys = torch.round(xt[:, 1] * scale) / scale
        uniq_keys, inv_idx = torch.unique(t_keys, return_inverse=True)

        params = []
        uniq_list = uniq_keys.detach().cpu().tolist()
        for tv in uniq_list:
            if self.out_dim == 1:
                if self._pde_name == "KdV":
                    params.append(self._compute_kdv_alpha(float(tv)))
                elif self._pde_name == "KS":
                    params.append(self._compute_ks_alpha(float(tv)))
                else:
                    params.append(self._compute_kdv_alpha(float(tv)))
            else:
                params.append(self._compute_nls_scale(float(tv)))

        params = torch.tensor(params, device=DEVICE, dtype=u_raw.dtype).view(-1, 1)
        param_per_row = params[inv_idx]

        if self.out_dim == 1:
            return u_raw + param_per_row
        else:
            return u_raw * param_per_row

    def forward(self, xt):
        if self._proj_enabled and self.pde_type in ("conservative", "mean_preserving"):
            return self.forward_proj(xt)
        return self.forward_raw(xt)

    # ✅ PATCH 1: 防御性强制关闭投影，确保训练阶段绝不使用投影（Two-Stage 核心思想）
    def compute_loss(self, x_f, t_f, x_ic, ts, residual_fn, ic_fn, inv_fn, inv0, warmup, tau):
        proj_was_enabled = self._proj_enabled
        if proj_was_enabled:
            self._proj_enabled = False
        
        try:
            Lp, _ = causal_pde_loss(self, x_f, t_f, residual_fn, tau)
            Lic = ic_loss(self, x_ic, ic_fn, self.out_dim)
            Ltot = Lp + IC_WEIGHT_VANILLA * Lic
            zero = torch.tensor(0.0, device=DEVICE)
            return Ltot, dict(Lp=Lp, Lic=Lic, Cinv=zero, Linv=zero, Ltot=Ltot)
        finally:
            if proj_was_enabled:
                self._proj_enabled = True


# ==============================================================================
# CHECKPOINT & RESULT MANAGEMENT (FULL BREAKPOINT RESUMPTION)
# ==============================================================================
def get_checkpoint_path(pde_name, method_label, seed):
    safe = method_label.replace(" ", "_")
    T_key = f"T{CONFIG.T_FINAL:.2f}".replace(".", "p")
    return os.path.join(CHECKPOINT_DIR, f"ckpt_{pde_name}_{safe}_{T_key}_seed{seed}.pth")


def get_reference_path(pde_name):
    T_key = f"T{CONFIG.T_FINAL:.2f}".replace(".", "p")
    return os.path.join(RESULT_DIR, f"ref_{pde_name}_{T_key}.npz")


def get_stats_path(pde_name):
    T_key = f"T{CONFIG.T_FINAL:.2f}".replace(".", "p")
    return os.path.join(RESULT_DIR, f"stats_{pde_name}_{T_key}.json")


def get_ablation_path():
    return os.path.join(RESULT_DIR, "ablation_table.json")


def save_checkpoint(pde_name, method_label, seed, model, optimizers, scheduler, current_iter, loss_rec):
    checkpoint = {
        "model_state": model.state_dict(),
        "current_iter": current_iter,
        "loss_rec": dataclasses.asdict(loss_rec),
        "method": method_label,
        "pde": pde_name,
        "seed": seed,
        "T_FINAL": CONFIG.T_FINAL,
    }
    if isinstance(optimizers, (list, tuple)):
        checkpoint["opt_states"] = [opt.state_dict() for opt in optimizers]
    else:
        checkpoint["opt_states"] = [optimizers.state_dict()] if optimizers else None
    if scheduler:
        checkpoint["scheduler_state"] = scheduler.state_dict()
    torch.save(checkpoint, get_checkpoint_path(pde_name, method_label, seed))


def load_checkpoint(pde_name, method_label, seed, model, optimizers, scheduler):
    ckpt_path = get_checkpoint_path(pde_name, method_label, seed)
    if not os.path.exists(ckpt_path):
        log.debug(f"No checkpoint found at {ckpt_path}, starting from scratch")
        return 0, LossRecord()

    checkpoint = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    
    if "T_FINAL" in checkpoint and abs(checkpoint["T_FINAL"] - CONFIG.T_FINAL) > 1e-6:
        log.warning(f"Checkpoint T_FINAL={checkpoint['T_FINAL']} != CONFIG T_FINAL={CONFIG.T_FINAL}, starting from scratch")
        return 0, LossRecord()
    
    model.load_state_dict(checkpoint["model_state"], strict=False)

    if checkpoint.get("opt_states") and optimizers is not None:
        if isinstance(optimizers, (list, tuple)):
            for opt, s in zip(optimizers, checkpoint["opt_states"]):
                opt.load_state_dict(s)
        else:
            optimizers.load_state_dict(checkpoint["opt_states"][0])

    if scheduler and "scheduler_state" in checkpoint:
        scheduler.load_state_dict(checkpoint["scheduler_state"])

    current_iter = checkpoint.get("current_iter", 0)
    loss_rec = LossRecord(**checkpoint.get("loss_rec", {}))
    
    if current_iter >= ITERATIONS:
        log.info(f"✓ Seed {seed} already completed ({current_iter}/{ITERATIONS} iterations), skipping training")
    else:
        log.info(f"⧑ Resuming seed {seed} from iteration {current_iter}")
    
    return current_iter, loss_rec


def load_existing_stats(pde_name):
    """加载已有的统计结果，避免重复计算"""
    stats_path = get_stats_path(pde_name)
    if not os.path.exists(stats_path):
        return {}
    
    try:
        with open(stats_path, "r") as f:
            existing_stats = json.load(f)
        log.info(f"✓ Loaded existing results for {len(existing_stats)} methods from {stats_path}")
        return existing_stats
    except Exception as e:
        log.warning(f"Failed to load existing stats: {e}, starting fresh")
        return {}


def save_method_stats(pde_name, method_label, stats):
    """保存单个方法的统计结果到JSON文件"""
    stats_path = get_stats_path(pde_name)
    
    all_stats = load_existing_stats(pde_name)
    all_stats[method_label] = stats.get_summary_table()
    
    with open(stats_path, "w") as f:
        json.dump(all_stats, f, indent=2)
    
    log.info(f"✓ Saved results for {method_label} to {stats_path}")

# ======================
# ✅ 最小新增：保存损失曲线到 JSON
# ======================
def save_loss_curve(pde_name, method_label, seed, loss_rec):
    safe = method_label.replace(" ", "_")
    path = os.path.join(CURVES_DIR, f"loss_{pde_name}_{safe}_seed{seed}.json")
    data = {
        "Lp": loss_rec.Lp,
        "Lic": loss_rec.Lic,
        "Cinv": loss_rec.Cinv,
        "Linv": loss_rec.Linv,
        "Ltot": loss_rec.Ltot,
        "lam_ic": loss_rec.lam_ic,
        "lam_inv": loss_rec.lam_inv,
    }
    with open(path, "w") as f:
        json.dump(data, f, indent=2)


# ==============================================================================
# STATISTICS FRAMEWORK (PATCH 6: KEY UNIFICATION)
# ==============================================================================
@dataclasses.dataclass
class MethodStatistics:
    label: str
    pde_name: str
    l2_errors: List[float] = dataclasses.field(default_factory=list)
    l2_errors_proj: List[float] = dataclasses.field(default_factory=list)
    linf_errors: List[float] = dataclasses.field(default_factory=list)
    linf_errors_proj: List[float] = dataclasses.field(default_factory=list)
    rel_l2_errors: List[float] = dataclasses.field(default_factory=list)
    rel_l2_errors_proj: List[float] = dataclasses.field(default_factory=list)
    residuals_raw: List[float] = dataclasses.field(default_factory=list)
    residuals_proj: List[float] = dataclasses.field(default_factory=list)
    constraint_errors_raw: List[float] = dataclasses.field(default_factory=list)
    constraint_errors_proj: List[float] = dataclasses.field(default_factory=list)
    projection_stats: List[Dict] = dataclasses.field(default_factory=list)
    train_time_seconds: List[float] = dataclasses.field(default_factory=list)
    peak_gpu_memory_gb: List[float] = dataclasses.field(default_factory=list)

    def get_summary_table(self) -> Dict[str, float]:
        summary = {
            "label": self.label,
            "L2_mean": float(np.mean(self.l2_errors)) if self.l2_errors else None,
            "L2_std": float(np.std(self.l2_errors)) if self.l2_errors else None,
            "L2_proj_mean": float(np.mean(self.l2_errors_proj)) if self.l2_errors_proj else None,
            "L2_proj_std": float(np.std(self.l2_errors_proj)) if self.l2_errors_proj else None,
            "Linf_mean": float(np.mean(self.linf_errors)) if self.linf_errors else None,
            "Linf_std": float(np.std(self.linf_errors)) if self.linf_errors else None,
            "Linf_proj_mean": float(np.mean(self.linf_errors_proj)) if self.linf_errors_proj else None,
            "Linf_proj_std": float(np.std(self.linf_errors_proj)) if self.linf_errors_proj else None,
            "Rel_L2_mean": float(np.mean(self.rel_l2_errors)) if self.rel_l2_errors else None,
            "Rel_L2_std": float(np.std(self.rel_l2_errors)) if self.rel_l2_errors else None,
            "Rel_L2_proj_mean": float(np.mean(self.rel_l2_errors_proj)) if self.rel_l2_errors_proj else None,
            "Rel_L2_proj_std": float(np.std(self.rel_l2_errors_proj)) if self.rel_l2_errors_proj else None,
            "Residual_raw_mean": float(np.mean(self.residuals_raw)) if self.residuals_raw else None,
            "Residual_raw_std": float(np.std(self.residuals_raw)) if self.residuals_raw else None,
            "Residual_proj_mean": float(np.mean(self.residuals_proj)) if self.residuals_proj else None,
            "Residual_proj_std": float(np.std(self.residuals_proj)) if self.residuals_proj else None,
            "Constraint_raw_mean": float(np.mean(self.constraint_errors_raw)) if self.constraint_errors_raw else None,
            "Constraint_raw_std": float(np.std(self.constraint_errors_raw)) if self.constraint_errors_raw else None,
            "Constraint_proj_mean": float(np.mean(self.constraint_errors_proj)) if self.constraint_errors_proj else None,
            "Constraint_proj_std": float(np.std(self.constraint_errors_proj)) if self.constraint_errors_proj else None,
            "Train_time_mean": float(np.mean(self.train_time_seconds)) if self.train_time_seconds else None,
            "Train_time_std": float(np.std(self.train_time_seconds)) if self.train_time_seconds else None,
            "Peak_GPU_GB_mean": float(np.mean(self.peak_gpu_memory_gb)) if self.peak_gpu_memory_gb else None,
            "Peak_GPU_GB_std": float(np.std(self.peak_gpu_memory_gb)) if self.peak_gpu_memory_gb else None,
        }
        
        # ✅ PATCH 6: 强制初始化投影参数键为 None，确保下游键结构统一
        summary.setdefault("Mean_abs_alpha_mean", None)
        summary.setdefault("Max_abs_alpha_mean", None)
        summary.setdefault("Mean_abs_s_minus_1_mean", None)
        summary.setdefault("Max_abs_s_minus_1_mean", None)

        if self.projection_stats:
            if "mean_abs_alpha" in self.projection_stats[0]:
                summary["Mean_abs_alpha_mean"] = float(np.mean([s["mean_abs_alpha"] for s in self.projection_stats]))
                summary["Max_abs_alpha_mean"] = float(np.mean([s["max_abs_alpha"] for s in self.projection_stats]))
            elif "mean_abs_s_minus_1" in self.projection_stats[0]:
                summary["Mean_abs_s_minus_1_mean"] = float(np.mean([s["mean_abs_s_minus_1"] for s in self.projection_stats]))
                summary["Max_abs_s_minus_1_mean"] = float(np.mean([s["max_abs_s_minus_1"] for s in self.projection_stats]))
        
        return summary


# ==============================================================================
# REFERENCE SOLVER (ETDRK4)
# ==============================================================================
def spectral_grid(nx=NX_REF):
    x = np.linspace(CONFIG.X_MIN, CONFIG.X_MAX, nx, endpoint=False)
    L = CONFIG.X_MAX - CONFIG.X_MIN
    k = (2.0 * np.pi / L) * np.fft.fftfreq(nx, d=L / nx)
    return x, k


def etdrk4_coeffs(Lk, dt):
    E = np.exp(dt * Lk)
    E2 = np.exp(dt * Lk / 2.0)
    M = 16
    r = np.exp(1j * np.pi * (np.arange(1, M + 1) - 0.5) / M)
    LR = dt * Lk[:, None] + r[None, :]
    Q = dt * np.real(np.mean((np.exp(LR / 2.0) - 1.0) / LR, axis=1))
    f1 = dt * np.real(np.mean((-4 - LR + np.exp(LR)*(4 - 3*LR + LR**2)) / LR**3, axis=1))
    f2 = dt * np.real(np.mean((2 + LR + np.exp(LR)*(-2 + LR)) / LR**3, axis=1))
    f3 = dt * np.real(np.mean((-4 - 3*LR - LR**2 + np.exp(LR)*(4 - LR)) / LR**3, axis=1))
    return E, E2, Q, f1, f2, f3


def solve_reference(pde_name: str, out_dim: int):
    x, k = spectral_grid(NX_REF)
    dt = DT_REF_KS if pde_name == "KS" else DT_REF_DEFAULT
    nsteps = int(round(CONFIG.T_FINAL / dt))
    save_idx = np.arange(0, nsteps + 1, SAVE_EVERY)
    t_save = save_idx * dt
    if t_save[-1] < CONFIG.T_FINAL - 1e-12:
        save_idx = np.append(save_idx, nsteps)
        t_save = np.append(t_save, nsteps * dt)

    x_torch = torch.tensor(x, dtype=torch.float64)
    if pde_name == "KdV":
        u0 = (0.5 / torch.cosh(x_torch / 2.0) ** 2).numpy()
        u = u0.astype(np.complex128)
    elif pde_name == "Burgers":
        u0 = (-torch.sin(np.pi * x_torch / 10.0)).numpy()
        u = u0.astype(np.complex128)
    elif pde_name == "NLS":
        u0 = torch.exp(-x_torch**2 / 2.0).numpy()
        u = u0.astype(np.complex128)
    elif pde_name == "KS":
        u0 = torch.cos(np.pi * x_torch / 16.0).numpy()
        u = u0.astype(np.complex128)
    else:
        raise ValueError(pde_name)

    ik = 1j * k
    k2 = k**2

    if pde_name == "KdV":
        Lk = 1j * k**3
        def N(u_phys):
            ux = np.fft.ifft(ik * np.fft.fft(u_phys)).real
            return -6.0 * u_phys * ux
    elif pde_name == "Burgers":
        nu = PDE.NU
        Lk = -nu * k**2
        def N(u_phys):
            ux = np.fft.ifft(ik * np.fft.fft(u_phys)).real
            return -(u_phys * ux)
    elif pde_name == "NLS":
        Lk = 1j * (-0.5 * k2)
        def N(u_phys):
            return -1j * np.abs(u_phys)**2 * u_phys
    elif pde_name == "KS":
        Lk = k2 - k**4
        def N(u_phys):
            ux = np.fft.ifft(ik * np.fft.fft(u_phys)).real
            return -(u_phys * ux)
    else:
        raise ValueError(pde_name)

    E, E2, Q, f1, f2, f3 = etdrk4_coeffs(Lk.astype(np.complex128), dt)
    u_save = np.zeros((len(t_save), NX_REF, out_dim), dtype=np.float64)
    save_ptr = 0

    def save_state(uc):
        nonlocal save_ptr
        if pde_name == "NLS":
            u_save[save_ptr, :, 0] = uc.real
            u_save[save_ptr, :, 1] = uc.imag
        else:
            u_save[save_ptr, :, 0] = uc.real
        save_ptr += 1

    save_state(u)
    v = np.fft.fft(u)
    for n in range(1, nsteps + 1):
        up = np.fft.ifft(v).real if pde_name != "NLS" else np.fft.ifft(v)
        Nv = np.fft.fft(N(up))
        a = E2*v + Q*Nv
        ua = np.fft.ifft(a).real if pde_name != "NLS" else np.fft.ifft(a)
        Na = np.fft.fft(N(ua))
        b = E2*v + Q*Na
        ub = np.fft.ifft(b).real if pde_name != "NLS" else np.fft.ifft(b)
        Nb = np.fft.fft(N(ub))
        c = E2*a + Q*(2*Nb-Nv)
        uc = np.fft.ifft(c).real if pde_name != "NLS" else np.fft.ifft(c)
        Nc = np.fft.fft(N(uc))
        v = E*v + Nv*f1 + (Na+Nb)*f2 + Nc*f3
        if n in save_idx:
            save_state(np.fft.ifft(v))

    return x, t_save, u_save


# ==============================================================================
# EVALUATION FUNCTIONS (PATCHED)
# ==============================================================================
def eval_reference_error(model, x_ref, t_ref, u_ref, out_dim, use_proj=False):
    model.eval()
    
    x_pt = torch.tensor(x_ref, device=DEVICE, dtype=torch.float32)
    t_pt = torch.tensor(t_ref, device=DEVICE, dtype=torch.float32)
    
    nx = x_pt.shape[0]
    nt = t_pt.shape[0]
    
    X = x_pt.view(1, nx).repeat(nt, 1)
    T = t_pt.view(nt, 1).repeat(1, nx)
    
    xt = torch.stack([X, T], dim=2).reshape(nt * nx, 2)
    
    with torch.no_grad():
        if use_proj and isinstance(model, TwoStageProjPINN):
            u_pred = model.forward_proj(xt).reshape(nt, nx, out_dim).cpu().numpy()
        else:
            u_pred = model(xt).reshape(nt, nx, out_dim).cpu().numpy()
    
    errs_l2, errs_linf, errs_rel_l2 = [], [], []
    for i in range(nt):
        u_true = u_ref[i]
        if out_dim == 1:
            e = u_pred[i, :, 0] - u_true[:, 0]
            abs_err = np.sqrt(np.mean(e**2))
            ref_norm = np.sqrt(np.mean(u_true[:, 0]**2))
            errs_l2.append(float(abs_err))
            errs_rel_l2.append(float(abs_err / (ref_norm + 1e-12)))
            errs_linf.append(float(np.max(np.abs(e))))
        else:
            er = u_pred[i, :, 0] - u_true[:, 0]
            ei = u_pred[i, :, 1] - u_true[:, 1]
            en = np.sqrt(er**2 + ei**2)
            abs_err = np.sqrt(np.mean(en**2))
            ref_norm = np.sqrt(np.mean(u_true[:, 0]**2 + u_true[:, 1]**2))
            errs_l2.append(float(abs_err))
            errs_rel_l2.append(float(abs_err / (ref_norm + 1e-12)))
            errs_linf.append(float(np.max(en)))
    
    return {
        "mean_l2": float(np.mean(errs_l2)),
        "std_l2": float(np.std(errs_l2)),
        "mean_rel_l2": float(np.mean(errs_rel_l2)),
        "std_rel_l2": float(np.std(errs_rel_l2)),
        "max_linf": float(np.max(errs_linf)),
        "mean_linf": float(np.mean(errs_linf)),
    }


# ✅ PATCH 2: 优化梯度上下文位置，减少 GPU 碎片与上下文切换开销
def eval_residual_oos(model, residual_fn, n=N_RESID_TEST, batch_size=500, use_proj=False):
    model.eval()
    chunks = []
    done = 0
    with torch.enable_grad():
        while done < n:
            m = min(batch_size, n - done)
            x = torch.rand(m, 1, device=DEVICE) * (CONFIG.X_MAX - CONFIG.X_MIN) + CONFIG.X_MIN
            t = torch.rand(m, 1, device=DEVICE) * CONFIG.T_FINAL
            xt = torch.cat([x, t], dim=1).requires_grad_(True)
            
            if use_proj and isinstance(model, TwoStageProjPINN):
                u = model.forward_proj(xt)
            else:
                u = model(xt)
            
            r = residual_fn(u, xt)
            r2 = (r**2).mean(dim=1).detach()
            chunks.append(r2)
            done += m
    
    # 统一在循环结束后清理，避免频繁碎片
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    
    r2_all = torch.cat(chunks, dim=0)
    return {
        "mean": float(r2_all.mean().item()),
        "std": float(r2_all.std().item()),
        "max": float(r2_all.max().item()),
        "p95": float(torch.quantile(r2_all, 0.95).item())
    }


def get_projection_stats(model, ts):
    """投影参数统计（审稿人要求）"""
    if not isinstance(model, TwoStageProjPINN) or not model._proj_enabled:
        return None
    params = []
    for t in ts:
        t_val = float(t)
        if model.out_dim == 1:
            if model._pde_name == "KdV":
                params.append(model._compute_kdv_alpha(t_val))
            elif model._pde_name == "KS":
                params.append(model._compute_ks_alpha(t_val))
            else:
                params.append(model._compute_kdv_alpha(t_val))
        else:
            params.append(model._compute_nls_scale(t_val))
    params_np = np.array(params)
    if model.out_dim == 1:
        return {
            "mean_abs_alpha": float(np.mean(np.abs(params_np))),
            "max_abs_alpha": float(np.max(np.abs(params_np))),
            "std_alpha": float(np.std(params_np))
        }
    else:
        return {
            "mean_abs_s_minus_1": float(np.mean(np.abs(params_np - 1.0))),
            "max_abs_s_minus_1": float(np.max(np.abs(params_np - 1.0))),
            "std_s": float(np.std(params_np))
        }


def eval_constraint_error(model, pde_type, inv_fn, inv0, use_proj=False):
    if pde_type == "dissipative":
        model.eval()
        norm = abs(inv0) + 1e-12
        resid_list = []
        with torch.enable_grad():
            for t in np.linspace(0.0, CONFIG.T_FINAL, N_CONSERVATION_PTS):
                x = quad_grid(CONFIG.N_INV_QUAD, requires_grad=False)
                t_leaf = torch.tensor(float(t), device=DEVICE, requires_grad=True)
                xt = torch.cat([x, t_leaf.expand_as(x)], dim=1).requires_grad_(True)
                if use_proj and isinstance(model, TwoStageProjPINN):
                    u = model.forward_proj(xt)
                else:
                    u = model(xt)
                E = PDE.burgers_E(u, xt)
                D = PDE.burgers_D(u, xt)
                # 评估阶段不需要二阶梯度，create_graph=False 正确
                dEdt = autograd.grad(E, t_leaf, create_graph=False)[0]
                r = dEdt + D
                resid_list.append(float((r / norm).abs().item()))
        return {
            "mean": float(np.mean(resid_list)),
            "max": float(np.max(resid_list)),
        }

    model.eval()
    norm = abs(inv0) + 1e-12
    errs = []
    with torch.enable_grad():
        for t in np.linspace(0.0, CONFIG.T_FINAL, N_CONSERVATION_PTS):
            x = quad_grid(CONFIG.N_INV_QUAD, requires_grad=False)
            xt = torch.cat([x, float(t) * torch.ones_like(x)], dim=1).requires_grad_(True)
            if use_proj and isinstance(model, TwoStageProjPINN):
                u = model.forward_proj(xt)
            else:
                u = model(xt)
            I = inv_fn(u, xt)
            c = torch.abs((I - inv0) / norm)
            errs.append(c.detach())

    errs = torch.stack(errs)
    return {
        "mean": float(errs.mean().item()),
        "max": float(errs.max().item()),
    }


# ==============================================================================
# TRAIN LOOP (PATCHED)
# ==============================================================================
def train_model(model, residual_fn, ic_fn, inv_fn, inv0, tau, pde_name, seed, desc=""):
    set_seed(seed)
    
    train_start_time = time.time()
    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    
    if isinstance(model, SAPINN_Pos):
        net_opt = torch.optim.Adam(model.net.parameters(), lr=LEARNING_RATE)
        dual_opt = torch.optim.Adam([model.log_lam_ic, model.log_lam_inv], lr=SA_LR)
        optimizers = [net_opt, dual_opt]
        scheduler = None
    else:
        opt = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
        optimizers = opt
        scheduler = torch.optim.lr_scheduler.MultiStepLR(opt, LR_MILESTONES, LR_GAMMA)

    start_iter, rec = load_checkpoint(pde_name, model.label, seed, model, optimizers, scheduler)
    
    if start_iter >= ITERATIONS:
        train_time = 0.0
        peak_gpu_memory = 0.0
        if DEVICE.type == "cuda":
            peak_gpu_memory = torch.cuda.max_memory_allocated() / (1024**3)
        return rec, model, train_time, peak_gpu_memory

    ts = time_slices(CONFIG.N_TS_INVARIANT)
    x_ic = sample_ic(CONFIG.N_IC_TRAIN)
    x_f, t_f = sample_colloc(CONFIG.N_TRAIN_COLLOC)

    pbar = tqdm(range(start_iter + 1, ITERATIONS + 1), desc=desc, ncols=120, leave=False)

    for it in pbar:
        warmup = it <= WARMUP_ITERS

        if CONFIG.RESAMPLE_INTERVAL > 0 and not warmup and (it - WARMUP_ITERS) % CONFIG.RESAMPLE_INTERVAL == 0:
            x_f, t_f = resample_collocation(model, x_f, t_f, residual_fn)

        model.train()

        if isinstance(model, SAPINN_Pos):
            loss_net, comps = model.compute_loss(
                x_f, t_f, x_ic, ts, residual_fn, ic_fn, inv_fn, inv0, warmup, tau
            )
            net_opt, dual_opt = optimizers
            net_opt.zero_grad()
            loss_net.backward()
            nn.utils.clip_grad_norm_(model.net.parameters(), 1.0)
            net_opt.step()

            if not warmup:
                lic_val = comps["Lic"].detach()
                c_inv = comps["Cinv"].detach()
                lam_ic = model.log_lam_ic.exp()
                lam_inv = model.log_lam_inv.exp()
                dual_loss = -(lam_ic * lic_val + lam_inv * c_inv)
                dual_opt.zero_grad()
                dual_loss.backward()
                # ✅ PATCH 4: 对偶参数梯度裁剪，防止 lam 震荡
                nn.utils.clip_grad_norm_([model.log_lam_ic, model.log_lam_inv], 1.0)
                dual_opt.step()
            else:
                dual_opt.zero_grad()

        else:
            opt = optimizers
            opt.zero_grad()
            loss, comps = model.compute_loss(
                x_f, t_f, x_ic, ts, residual_fn, ic_fn, inv_fn, inv0, warmup, tau
            )
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            if scheduler:
                scheduler.step()

        scalars = {}
        for k, v in comps.items():
            scalars[k] = float(v.item()) if torch.is_tensor(v) else v

        rec.Lp.append(float(scalars["Lp"]))
        rec.Lic.append(float(scalars["Lic"]))
        rec.Cinv.append(float(scalars["Cinv"]))
        rec.Linv.append(float(scalars["Linv"]))
        rec.Ltot.append(float(scalars["Ltot"]))

        if isinstance(model, SAPINN_Pos):
            rec.lam_ic.append(float(model.lam_ic))
            rec.lam_inv.append(float(model.lam_inv))
        else:
            rec.lam_ic.append(0.0)
            rec.lam_inv.append(0.0)

        if it % LOG_INTERVAL == 0:
            pbar.set_postfix(
                Lp=f"{float(scalars['Lp']):.1e}",
                Lic=f"{float(scalars['Lic']):.1e}",
                W="Y" if warmup else "N",
            )

        if it % SAVE_CHECKPOINT_INTERVAL == 0:
            save_checkpoint(pde_name, model.label, seed, model, optimizers, scheduler, it, rec)

    save_checkpoint(pde_name, model.label, seed, model, optimizers, scheduler, ITERATIONS, rec)
    
    train_time = time.time() - train_start_time
    peak_gpu_memory = 0.0
    if DEVICE.type == "cuda":
        peak_gpu_memory = torch.cuda.max_memory_allocated() / (1024**3)
    save_loss_curve(pde_name, model.label, seed, rec)
    
    return rec, model, train_time, peak_gpu_memory


# ==============================================================================
# MAIN (CRITICAL FIX FOR ABLATION TABLE KEY UNIFICATION)
# ==============================================================================
METHODS_REGISTRY = [
    VanillaPINN, PenaltyPINN, FixedLambdaPINN, SAPINN_Pos, TwoStageProjPINN,
]


def main():
    log.info("=" * 80)
    log.info(f"JCP FINAL v6.0.2 - PATCHED (ABLATION TABLE KEY UNIFICATION)")
    log.info(f"Device: {DEVICE}")
    log.info(f"Methods: {[m.label for m in METHODS_REGISTRY]}")
    log.info(f"Seeds: {CONFIG.SEEDS}")
    log.info(f"Iterations: {ITERATIONS}")
    log.info("=" * 80)

    all_ablation = {}

    for (pde_name, out_dim, pde_type) in PDE_LIST:
        log.info(f"\n{'='*80}")
        log.info(f"PDE: {pde_name} | Type: {pde_type} | Out dim: {out_dim}")
        log.info(f"{'='*80}\n")

        residual_fn, ic_fn, inv_fn = PDE_REGISTRY[pde_name]

        ref_path = get_reference_path(pde_name)
        if os.path.exists(ref_path):
            log.info(f"✓ Loading reference solution for {pde_name}")
            data = np.load(ref_path)
            x_ref, t_ref, u_ref = data["x"], data["t"], data["u"]
        else:
            log.info(f"⧑ Solving reference with ETDRK4 (T_FINAL={CONFIG.T_FINAL})")
            x_ref, t_ref, u_ref = solve_reference(pde_name, out_dim)
            np.savez_compressed(ref_path, x=x_ref, t=t_ref, u=u_ref)
            log.info(f"✓ Reference saved to {ref_path}")

        x0 = quad_grid(CONFIG.N_INV_QUAD, requires_grad=False)

        if pde_name == "KdV":
            inv0 = PDE.kdv_I_ic(x0)
        elif pde_name == "NLS":
            inv0 = PDE.nls_I_ic(x0)
        elif pde_name == "KS":
            inv0 = PDE.ks_I_ic(x0)
        else:
            xt0 = torch.cat([x0, torch.zeros_like(x0)], dim=1)
            u0 = PDE.burgers_ic(x0)
            inv0 = float(PDE.burgers_E(u0, xt0).item())

        log.info(f"I0 = {inv0:.4e}")
        tau = PDE_CAUSAL_TAU[pde_name]

        existing_stats = load_existing_stats(pde_name)
        method_stats_dict: Dict[str, MethodStatistics] = {}
        pde_ablation = {}

        # ✅ FIX 1: 首先将所有已存在的结果添加到pde_ablation中
        for method_label, existing_data in existing_stats.items():
            log.info(f"✓ [{method_label:18s}] Loading existing results from stats file")
            
            # 将现有数据转换为ablation_entry格式
            ablation_entry = {
                "mean_l2_raw": existing_data.get("L2_mean"),
                "std_l2_raw": existing_data.get("L2_std"),
                "mean_rel_l2_raw": existing_data.get("Rel_L2_mean"),
                "mean_linf_raw": existing_data.get("Linf_mean"),
                "mean_residual_raw": existing_data.get("Residual_raw_mean"),
                "mean_constraint_raw": existing_data.get("Constraint_raw_mean"),
                "train_time_mean": existing_data.get("Train_time_mean"),
                "peak_gpu_mean": existing_data.get("Peak_GPU_GB_mean"),
            }
            
            # 添加投影相关数据（如果存在）
            if "L2_proj_mean" in existing_data and existing_data["L2_proj_mean"] is not None:
                ablation_entry.update({
                    "mean_l2_proj": existing_data.get("L2_proj_mean"),
                    "std_l2_proj": existing_data.get("L2_proj_std"),
                    "mean_rel_l2_proj": existing_data.get("Rel_L2_proj_mean"),
                    "mean_linf_proj": existing_data.get("Linf_proj_mean"),
                    "mean_residual_proj": existing_data.get("Residual_proj_mean"),
                    "mean_constraint_proj": existing_data.get("Constraint_proj_mean"),
                })
                
                if existing_data.get("Residual_proj_mean") and existing_data.get("Residual_raw_mean"):
                    ablation_entry["residual_ratio"] = existing_data["Residual_proj_mean"] / existing_data["Residual_raw_mean"]
                
                if "Mean_abs_alpha_mean" in existing_data and existing_data["Mean_abs_alpha_mean"] is not None:
                    ablation_entry.update({
                        "mean_abs_alpha": existing_data.get("Mean_abs_alpha_mean"),
                        "max_abs_alpha": existing_data.get("Max_abs_alpha_mean"),
                    })
                elif "Mean_abs_s_minus_1_mean" in existing_data and existing_data["Mean_abs_s_minus_1_mean"] is not None:
                    ablation_entry.update({
                        "mean_abs_s_minus_1": existing_data.get("Mean_abs_s_minus_1_mean"),
                        "max_abs_s_minus_1": existing_data.get("Max_abs_s_minus_1_mean"),
                    })
            
            # ✅ PATCH 6: 保持键结构完整，不过滤 None，确保下游统一
            pde_ablation[method_label] = ablation_entry
            log.info(f"  └─ Successfully loaded {len(ablation_entry)} fields")

        for MethodCls in METHODS_REGISTRY:
            method_label = MethodCls.label
            
            # ✅ FIX 2: 只有当方法不在pde_ablation中时才运行
            if method_label in pde_ablation:
                log.info(f"✓ [{method_label:18s}] Already in ablation table, skipping")
                stats = MethodStatistics(label=method_label, pde_name=pde_name)
                method_stats_dict[method_label] = stats
                continue

            log.info(f"[{method_label:18s}] Running {CONFIG.NUM_SEEDS} seeds...")

            stats = MethodStatistics(
                label=method_label,
                pde_name=pde_name,
            )

            trained_model = None  # 初始化变量，避免未定义错误
            
            for seed_idx, seed in enumerate(CONFIG.SEEDS):
                log.info(f"  └─ Seed {seed_idx+1}/{CONFIG.NUM_SEEDS} (seed={seed})")

                set_seed(seed)
                
                if MethodCls is FixedLambdaPINN:
                    model = MethodCls(out_dim, pde_type, pde_name).to(DEVICE)
                elif MethodCls is TwoStageProjPINN:
                    model = MethodCls(out_dim, pde_type, pde_name).to(DEVICE)
                else:
                    model = MethodCls(out_dim, pde_type).to(DEVICE)

                rec, trained_model, train_time, peak_gpu = train_model(
                    model, residual_fn, ic_fn, inv_fn, inv0, tau,
                    pde_name, seed, f"{pde_name}|{method_label}|S{seed}"
                )

                proj_stats = None
                if isinstance(trained_model, TwoStageProjPINN) and pde_type in ("conservative", "mean_preserving"):
                    log.info("    └─ Enabling projection for evaluation...")
                    trained_model.enable_projection(inv_fn, inv0)
                    proj_stats = get_projection_stats(trained_model, time_slices(N_CONSERVATION_PTS))
                    stats.projection_stats.append(proj_stats)

                ref_dict_raw = eval_reference_error(trained_model, x_ref, t_ref, u_ref, out_dim, use_proj=False)
                resid_dict_raw = eval_residual_oos(trained_model, residual_fn, use_proj=False)
                c_err_raw = eval_constraint_error(trained_model, pde_type, inv_fn, inv0, use_proj=False)

                stats.l2_errors.append(ref_dict_raw["mean_l2"])
                stats.rel_l2_errors.append(ref_dict_raw["mean_rel_l2"])
                stats.linf_errors.append(ref_dict_raw["max_linf"])
                stats.residuals_raw.append(resid_dict_raw["mean"])
                stats.constraint_errors_raw.append(c_err_raw["mean"])
                stats.train_time_seconds.append(train_time)
                stats.peak_gpu_memory_gb.append(peak_gpu)

                if isinstance(trained_model, TwoStageProjPINN):
                    ref_dict_proj = eval_reference_error(trained_model, x_ref, t_ref, u_ref, out_dim, use_proj=True)
                    resid_dict_proj = eval_residual_oos(trained_model, residual_fn, use_proj=True)
                    c_err_proj = eval_constraint_error(trained_model, pde_type, inv_fn, inv0, use_proj=True)
                    
                    stats.l2_errors_proj.append(ref_dict_proj["mean_l2"])
                    stats.rel_l2_errors_proj.append(ref_dict_proj["mean_rel_l2"])
                    stats.linf_errors_proj.append(ref_dict_proj["max_linf"])
                    stats.residuals_proj.append(resid_dict_proj["mean"])
                    stats.constraint_errors_proj.append(c_err_proj["mean"])

                if seed_idx == 0:
                    log.info("    └─ Saving full field data...")
                    x_pt = torch.tensor(x_ref, device=DEVICE, dtype=torch.float32).unsqueeze(1)
                    t_pt = torch.tensor(t_ref, device=DEVICE, dtype=torch.float32).unsqueeze(1)
                    nt = t_pt.shape[0]
                    nx = x_pt.shape[0]
                    x_all = x_pt.repeat(nt, 1)
                    t_all = t_pt.repeat_interleave(nx, dim=0)
                    xt_all = torch.cat([x_all, t_all], dim=1)

                    pred_path = os.path.join(FIELDS_DIR, f"pred_{pde_name}_{method_label}.npz")

                    if isinstance(trained_model, TwoStageProjPINN):
                        with torch.no_grad():
                            u_raw_all = trained_model.forward_raw(xt_all).reshape(nt, nx, out_dim).cpu().numpy()

                        with torch.enable_grad():
                            chunk_size = 40000
                            u_proj_list = []
                            for start in range(0, xt_all.shape[0], chunk_size):
                                end = start + chunk_size
                                sub_xt = xt_all[start:end].detach().clone().requires_grad_(True)
                                sub_u = trained_model.forward_proj(sub_xt).detach()
                                u_proj_list.append(sub_u)
                            u_proj_all = torch.cat(u_proj_list, dim=0).reshape(nt, nx, out_dim).cpu().numpy()

                        np.savez_compressed(
                            pred_path,
                            x=x_ref,
                            t=t_ref,
                            u_raw=u_raw_all,
                            u_proj=u_proj_all
                        )
                    else:
                        with torch.no_grad():
                            u_pred_all = trained_model(xt_all).reshape(nt, nx, out_dim).cpu().numpy()
                        np.savez_compressed(
                            pred_path,
                            x=x_ref,
                            t=t_ref,
                            u=u_pred_all
                        )

                log_str = f"      └─ L2_raw={ref_dict_raw['mean_l2']:.3e} | Resid_raw={resid_dict_raw['mean']:.3e} | C_raw={c_err_raw['mean']:.3e}"
                if isinstance(trained_model, TwoStageProjPINN):
                    log_str += f"\n      └─ L2_proj={ref_dict_proj['mean_l2']:.3e} | Resid_proj={resid_dict_proj['mean']:.3e} | C_proj={c_err_proj['mean']:.3e}"
                    log_str += f"\n      └─ Resid_ratio={resid_dict_proj['mean']/resid_dict_raw['mean']:.3f}x"
                    if proj_stats:
                        if out_dim == 1:
                            log_str += f" | Max|α|={proj_stats['max_abs_alpha']:.3e}"
                        else:
                            log_str += f" | Max|s-1|={proj_stats['max_abs_s_minus_1']:.3e}"
                log_str += f" | Time={train_time:.1f}s"
                log.info(log_str)

            summary_table = stats.get_summary_table()
            log.info(f"\n[{method_label:18s}] SUMMARY:")
            log.info(f"  L2_raw:     {summary_table['L2_mean']:.3e} ± {summary_table['L2_std']:.3e}")
            log.info(f"  Rel_L2_raw: {summary_table['Rel_L2_mean']:.3e} ± {summary_table['Rel_L2_std']:.3e}")
            log.info(f"  Resid_raw:  {summary_table['Residual_raw_mean']:.3e} ± {summary_table['Residual_raw_std']:.3e}")
            log.info(f"  C_raw:      {summary_table['Constraint_raw_mean']:.3e} ± {summary_table['Constraint_raw_std']:.3e}")
            
            if isinstance(trained_model, TwoStageProjPINN):
                log.info(f"  L2_proj:    {summary_table['L2_proj_mean']:.3e} ± {summary_table['L2_proj_std']:.3e}")
                log.info(f"  Resid_proj: {summary_table['Residual_proj_mean']:.3e} ± {summary_table['Residual_proj_std']:.3e}")
                log.info(f"  C_proj:     {summary_table['Constraint_proj_mean']:.3e} ± {summary_table['Constraint_proj_std']:.3e}")
                if 'Mean_abs_alpha_mean' in summary_table and summary_table['Mean_abs_alpha_mean'] is not None:
                    log.info(f"  Mean|α|:    {summary_table['Mean_abs_alpha_mean']:.3e}")
                    log.info(f"  Max|α|:     {summary_table['Max_abs_alpha_mean']:.3e}")
                elif 'Mean_abs_s_minus_1_mean' in summary_table and summary_table['Mean_abs_s_minus_1_mean'] is not None:
                    log.info(f"  Mean|s-1|:  {summary_table['Mean_abs_s_minus_1_mean']:.3e}")
                    log.info(f"  Max|s-1|:   {summary_table['Max_abs_s_minus_1_mean']:.3e}")
            
            log.info(f"  Time:       {summary_table['Train_time_mean']:.1f}s ± {summary_table['Train_time_std']:.1f}s")
            log.info(f"  GPU Mem:    {summary_table['Peak_GPU_GB_mean']:.2f}GB\n")

            method_stats_dict[method_label] = stats
            save_method_stats(pde_name, method_label, stats)
            
            ablation_entry = {
                "mean_l2_raw": summary_table["L2_mean"],
                "std_l2_raw": summary_table["L2_std"],
                "mean_rel_l2_raw": summary_table["Rel_L2_mean"],
                "mean_linf_raw": summary_table["Linf_mean"],
                "mean_residual_raw": summary_table["Residual_raw_mean"],
                "mean_constraint_raw": summary_table["Constraint_raw_mean"],
                "train_time_mean": summary_table["Train_time_mean"],
                "peak_gpu_mean": summary_table["Peak_GPU_GB_mean"],
            }
            
            if isinstance(trained_model, TwoStageProjPINN):
                raw_mean = summary_table["Residual_raw_mean"]
                proj_mean = summary_table["Residual_proj_mean"]
                ablation_entry.update({
                    "mean_l2_proj": summary_table["L2_proj_mean"],
                    "std_l2_proj": summary_table["L2_proj_std"],
                    "mean_rel_l2_proj": summary_table["Rel_L2_proj_mean"],
                    "mean_linf_proj": summary_table["Linf_proj_mean"],
                    "mean_residual_proj": proj_mean,
                    "mean_constraint_proj": summary_table["Constraint_proj_mean"],
                    "residual_ratio": (proj_mean / raw_mean) if raw_mean else None,
                })
                if summary_table.get("Mean_abs_alpha_mean") is not None:
                    ablation_entry.update({
                        "mean_abs_alpha": summary_table["Mean_abs_alpha_mean"],
                        "max_abs_alpha": summary_table["Max_abs_alpha_mean"],
                    })
                elif summary_table.get("Mean_abs_s_minus_1_mean") is not None:
                    ablation_entry.update({
                        "mean_abs_s_minus_1": summary_table["Mean_abs_s_minus_1_mean"],
                        "max_abs_s_minus_1": summary_table["Max_abs_s_minus_1_mean"],
                    })
            else:
                # ✅ PATCH 6: 非 TwoStage 方法统一填充 None，确保所有方法键结构一致
                ablation_entry.update({
                    "mean_l2_proj": None,
                    "std_l2_proj": None,
                    "mean_rel_l2_proj": None,
                    "mean_linf_proj": None,
                    "mean_residual_proj": None,
                    "mean_constraint_proj": None,
                    "residual_ratio": None,
                    "mean_abs_alpha": None,
                    "max_abs_alpha": None,
                    "mean_abs_s_minus_1": None,
                    "max_abs_s_minus_1": None,
                })
            
            pde_ablation[method_label] = ablation_entry

        # ✅ FIX 3: 打印最终的pde_ablation内容，用于调试
        log.info(f"\n📋 Final ablation table for {pde_name}:")
        log.info(f"   Methods: {list(pde_ablation.keys())}")
        for method_label, data in pde_ablation.items():
            log.info(f"     {method_label}: {len(data)} fields (all methods have identical keys)")

        all_ablation[pde_name] = pde_ablation

    with open(get_ablation_path(), "w", encoding="utf-8") as f:
        json.dump(all_ablation, f, indent=2, ensure_ascii=False)
    log.info(f"✓ Ablation table saved to {get_ablation_path()}")

    # ✅ FIX 4: 打印最终的all_ablation内容，用于调试
    log.info(f"\n📋 Final complete ablation table:")
    for pde_name, methods in all_ablation.items():
        log.info(f"  {pde_name}: {list(methods.keys())}")
        first_method = next(iter(methods.keys()))
        log.info(f"    键数量: {len(methods[first_method])} (所有方法一致)")

    log.info(f"\n{'='*80}")
    log.info("✅ ALL EXPERIMENTS COMPLETED SUCCESSFULLY!")
    log.info(f"Results directory: {RESULT_DIR}")
    log.info(f"Checkpoints: {CHECKPOINT_DIR}")
    log.info(f"Field data: {FIELDS_DIR}")
    log.info(f"{'='*80}\n")
    
    log.info("✅ PATCHES APPLIED IN v6.0.2:")
    log.info("✅ PATCH 1: TwoStageProjPINN.compute_loss defensive projection disable")
    log.info("✅ PATCH 2: eval_residual_oos gradient context optimization")
    log.info("✅ PATCH 3: KdV Newton iteration intermediate clamping")
    log.info("✅ PATCH 4: SAPINN_Pos dual parameter gradient clipping")
    log.info("✅ PATCH 5: Ablation table generation (critical fix for missing data)")
    log.info("✅ PATCH 6: Ablation table key structure unification (all methods identical keys)")
    log.info("✅ UNCHANGED: constraint_burgers create_graph=True (training requires it)")
    log.info(f"{'='*80}\n")


if __name__ == "__main__":
    main()

2026-05-24 00:26:57,497 INFO     ================================================================================
2026-05-24 00:26:57,497 INFO     JCP FINAL v6.0.2 - PATCHED (ABLATION TABLE KEY UNIFICATION)
2026-05-24 00:26:57,498 INFO     Device: cuda
2026-05-24 00:26:57,498 INFO     Methods: ['Vanilla', 'Penalty', 'Fixed-λ', 'SA-PINN-pos', 'TwoStage-Proj']
2026-05-24 00:26:57,498 INFO     Seeds: [42, 43, 44, 45, 46]
2026-05-24 00:26:57,498 INFO     Iterations: 2000
2026-05-24 00:26:57,499 INFO     ================================================================================
2026-05-24 00:26:57,499 INFO     
2026-05-24 00:26:57,499 INFO     PDE: KdV | Type: conservative | Out dim: 1
2026-05-24 00:26:57,499 INFO     ================================================================================

2026-05-24 00:26:57,500 INFO     ⧑ Solving reference with ETDRK4 (T_FINAL=1.0)
2026-05-24 00:27:00,418 INFO     ✓ Reference saved to jcp_results_v6_0_3/ref_KdV_T1p00.npz
2026-05-24 00:27:00